# Day 08 · Utopia — Chicago, 2038

**Question.** CTA promises that every 'L' station will be vertically accessible by 2038, and the Red
Line Extension will reach 130th Street in 2030. When that city exists, who can reach a station
without stairs, and how much of the promise has a date and money behind it?

**Approach.** Start from Plate III's step-free half-mile walk to accessible stations. Make the
remaining stations accessible in the order of CTA's All Stations Accessibility Program (2024 plan,
Table 5): Phase One (funded, 2027), Phase Two's funded stations (2028), the Red Line Extension (2030),
then the stations with no date or no money (Phase Two unfunded, Phases Three and Four, the future
Red–Purple Modernization). Rerun the walk after each step and count the residents (2020 Census
blocks) who gain it. The finished promise is drawn as land; the rest of the city as sea, with a line
for every eighth of a mile farther to walk.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | Stop snapshot, ASAP phase table, Red Line Extension from OpenStreetMap, 'L' track |
| 3. Validate | Station counts, every phase row matches one station, 36 left = CTA's 110 of 146 |
| 4. Transform | Stations with phase and year; step-free network |
| 5. Analyze | Cumulative walks by step, gains by phase and station, the finding |
| 6. Polygons | Strata (today, dated, no date) and waterlines |
| 7. QA view | Quick map and tables |
| 8. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import subprocess
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree
from skimage import measure

from atlas import Ledger, gtfs, snapshot, to_address, walk, write_plate

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('osmnx').setLevel(logging.WARNING)
log = logging.getLogger('day08')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = pull a new stop snapshot from the Chicago Data Portal

DAY = 8
STOPS_ID = '8pix-ypme'  # CTA - System Information - List of 'L' Stops
STOPS_NAME = "CTA - System Information - List of 'L' Stops"
STOPS_URL = f'https://data.cityofchicago.org/d/{STOPS_ID}'
ASAP_PLAN = 'https://www.transitchicago.com/assets/1/6/ASAP_Strategic_Plan_508_FINAL.pdf'
ASAP_PAGE = 'https://www.transitchicago.com/accessibility/asap/'
RLE_FAQ = 'https://www.transitchicago.com/rle/faq/'

DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
OSM_DIR = REPO / walk.OSM_DIR
BLOCKS_CSV = REPO / 'reference' / 'chicago-blocks-2020.csv'
BOUNDARY_RAW = sorted((REPO / 'reference' / 'raw').glob('city_boundary_*.geojson'))[-1]
assert (DAY_DIR / 'day08.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'

LINES = ('Red', 'Blue', 'Brown', 'Green', 'Orange', 'Purple', 'Pink', 'Yellow')
CTA_PUBLISHED = {'accessible': 110, 'total': 146}  # CTA, ASAP page and Austin release (7 Aug 2026)
PORTAL_COUNTS = {'total': 144, 'accessible': 106}  # Plates I, III and VII, same dataset

# Walk settings, as Day 03
WALK_M = walk.HALF_MILE_M
NETWORK_BUFFER_M = 1609.344
MAX_SNAP_M = 150
FRONTAGE_M = 40
MIN_HOLE_M2 = 20_000
MILE_M = 1609.344

# The steps toward 2038, in order: key, label, year (None = no date), funded
STEPS = [
    ('P1', 'Phase One', 2027, True),
    ('P2f', 'Phase Two, funded', 2028, True),
    ('RLE', 'Red Line Extension', 2030, True),
    ('P2u', 'Phase Two, unfunded', None, False),
    ('P3', 'Phase Three', None, False),
    ('P4', 'Phase Four', None, False),
    ('RPM', 'Red–Purple Modernization', None, False),
]
PHASE_KEY = {('Phase One', 'yes'): 'P1', ('Phase Two', 'yes'): 'P2f', ('Phase Two', 'no'): 'P2u',
             ('Phase Three', 'no'): 'P3', ('Phase Four', 'no'): 'P4', ('Red-Purple Modernization', 'no'): 'RPM'}

# Red Line Extension: the four stations (CTA RLE FAQ) where OpenStreetMap's construction alignment
# crosses the street CTA names for each main entrance
RLE_STATIONS = [('rle103', '103rd', '103rd Street'), ('rle111', '111th', '111th Street'),
                ('rleMich', 'Michigan', 'Michigan Avenue'), ('rle130', '130th', '130th Street')]

# Waterlines: the step-free walk beyond the half mile, one line per eighth of a mile, to four miles
WATER_LEVELS_MI = [0.5 + i / 8 for i in range(1, 29)]
WATER_CELL_M = 40

## 2. Pull

- **Stops:** dated snapshot of CTA's stop list (same file as Plate VII).
- **Phases:** CTA's ASAP Strategic Plan (2024), Table 5, typed into `data/raw/asap_phases_2024.csv` with
  its source; two rows mark stations finished since (Racine, Blue; Austin, Green), from CTA releases.
- **Red Line Extension:** the alignment as mapped in OpenStreetMap (`railway=construction`, "Red Line
  Extension") and the streets of the four stations, cut from the dated Geofabrik extract and saved
  in `data/raw/`.
- **Track:** CTA GTFS rail shapes, as Plates II, V and VII.

In [ ]:
stops_raw, stops_pulled = snapshot(STOPS_ID, RAW_DIR, 'l_stops', refresh=REFRESH,
                                   select='stop_id,map_id,station_descriptive_name,ada,location')
phases = pd.read_csv(RAW_DIR / 'asap_phases_2024.csv', dtype=str, keep_default_na=False)
blocks = pd.read_csv(BLOCKS_CSV, dtype={'geoid': str})
city = shapely.union_all(gpd.read_file(BOUNDARY_RAW).to_crs(walk.CRS).geometry)
boundary_pulled = BOUNDARY_RAW.stem.rsplit('_', 1)[-1]
pbf, osm_date = walk.latest_pbf(OSM_DIR)
log.info('Stops %s (%d rows); phases %d rows; OSM extract %s', stops_pulled, len(stops_raw), len(phases), osm_date)

In [ ]:
# The Red Line Extension and its station streets from OpenStreetMap, saved once per extract
rle_raw = RAW_DIR / f'osm_rle_{osm_date.isoformat()}.geojson'
if not rle_raw.exists():
    tmp = DAY_DIR / 'data' / 'processed'
    tmp.mkdir(parents=True, exist_ok=True)
    bbox = '-87.66,41.64,-87.57,41.73'  # 95th Street to 130th, Halsted to the Calumet River
    cut = tmp / 'rle_area.osm.pbf'
    subprocess.run(['osmium', 'extract', '-O', '-b', bbox, str(pbf), '-o', str(cut)], check=True)
    ways = tmp / 'rle_ways.osm.pbf'
    subprocess.run(['osmium', 'tags-filter', '-O', str(cut), 'w/railway=construction', 'w/highway', '-o', str(ways)], check=True)
    seq = tmp / 'rle_ways.geojsonseq'
    subprocess.run(['osmium', 'export', '-O', '-f', 'geojsonseq', str(ways), '-o', str(seq)], check=True)
    feats = []
    for line in seq.read_text().split('\n'):
        if not line.strip('\x1e').strip():
            continue
        f = json.loads(line.strip('\x1e'))
        p = f['properties']
        keep = (p.get('railway') == 'construction' and p.get('name') == 'Red Line Extension') or \
               (p.get('highway') and any(p.get('name', '').endswith(s) for _, _, s in RLE_STATIONS))
        if keep and f['geometry']['type'] == 'LineString':
            feats.append({'type': 'Feature', 'geometry': f['geometry'],
                          'properties': {k: p.get(k) for k in ('name', 'railway', 'highway', 'operator')}})
    rle_raw.write_text(json.dumps({'type': 'FeatureCollection', 'features': feats}))
osm_rle = gpd.read_file(rle_raw)
log.info('OSM features for the Red Line Extension: %s', osm_rle.assign(kind=osm_rle['railway'].fillna('street'))['kind'].value_counts().to_dict())

In [ ]:
# The 'L' track: CTA's GTFS rail shapes carrying at least a tenth of their line's trips (as Plate VII)
gtfs.RAW_DIR = REPO / gtfs.RAW_DIR
GTFS_ZIP, gtfs_date = gtfs.feed()
RAIL_NAMES = {'Red': 'Red', 'Blue': 'Blue', 'Brn': 'Brown', 'G': 'Green', 'Org': 'Orange', 'P': 'Purple', 'Pink': 'Pink', 'Y': 'Yellow'}
read = lambda f, **kw: pd.read_csv(zipfile.ZipFile(GTFS_ZIP).open(f), dtype=str, skipinitialspace=True, **kw)
rail_routes = read('routes.txt').query('route_type == "1"')
rail_trips = read('trips.txt', usecols=['route_id', 'shape_id'])
rail_counts = rail_trips[rail_trips['route_id'].isin(rail_routes['route_id'])].groupby(['route_id', 'shape_id']).size()
rail_counts = rail_counts[rail_counts >= 0.1 * rail_counts.groupby(level=0).transform('sum')]
rail_paths = gtfs.shape_paths(GTFS_ZIP, set(rail_counts.index.get_level_values(1)))
track = [{'line': RAIL_NAMES[rid],
          'path': [[round(x), round(y)] for x, y in shapely.simplify(shapely.LineString(rail_paths[sid][['x', 'y']].to_numpy()), 10).coords]}
         for rid, sid in rail_counts.index]
log.info('Rail shapes (GTFS feed of %s): %d', gtfs_date, len(track))

## 3. Validate

In [ ]:
EXPECTED = {'stop_id', 'map_id', 'station_descriptive_name', 'ada', 'location.latitude', 'location.longitude'}
checks = {
    'stop columns present': EXPECTED <= set(stops_raw.columns),
    'ADA flag is true/false': stops_raw['ada'].astype(str).str.lower().isin({'true', 'false'}).all(),
    'phase rows have a known phase': phases['phase'].isin({'done', 'Phase One', 'Phase Two', 'Phase Three', 'Phase Four', 'Red-Purple Modernization'}).all(),
    'Table 5 has 42 stations (2024)': (phases['phase'] != 'done').sum() + 6 == 42,  # 4 Red stations at Lawrence-Bryn Mawr are flagged accessible already
    'blocks add up to 2020 Chicago': blocks['pop'].sum() == 2_746_388,
    'OSM has the extension': (osm_rle['railway'] == 'construction').any(),
}
for name, ok in checks.items():
    log.info('%-36s %s', name, 'ok' if ok else 'FAILED')
assert all(checks.values()), 'Validation failed; see log above'

## 4. Transform

Stations as Plates I and III (accessible only if every platform is flagged accessible), then CTA's
newer word wins: Racine (Blue) and Austin (Green), reopened accessible, are counted accessible. Each
remaining station gets its phase from Table 5; the 36 left must match CTA's 110 of 146.

In [ ]:
import re
PAREN = re.compile(r'\((?P<inside>[^)]*)\)\s*$')
truth = lambda s: s.astype(str).str.lower().str.strip().eq('true')
stops = stops_raw.assign(map_id=stops_raw['map_id'].astype(str), ada=truth(stops_raw['ada']),
                         lat=pd.to_numeric(stops_raw['location.latitude']), lon=pd.to_numeric(stops_raw['location.longitude']))


def parse_lines(name: str) -> list[str]:
    m = PAREN.search(name)
    inside = m.group('inside') if m else ''
    return [line for line in LINES if re.search(rf'\b{line}\b', inside)]


stations = (stops.groupby('map_id', as_index=False)
            .agg(name=('station_descriptive_name', 'first'), flag=('ada', 'all'), lat=('lat', 'mean'), lon=('lon', 'mean')))
stations['name'] = stations['name'].str.replace(r'\s+', ' ', regex=True).str.strip()
stations['short'] = stations['name'].map(lambda s: PAREN.sub('', s).strip())
stations['lines'] = stations['name'].map(parse_lines)
counts = {'total': len(stations), 'accessible': int(stations['flag'].sum())}
assert counts == PORTAL_COUNTS, f'Station counts changed: {counts}'

stations['phase'] = np.where(stations['flag'], 'today', '')
stations['year'] = np.where(stations['flag'], 0, -1)
for r in phases.itertuples():
    m = (stations['short'] == r.station) & ~stations['flag']
    if r.line:
        m &= stations['lines'].map(lambda ls: r.line in ls)
    assert m.sum() == 1, f'{r.station} ({r.line}) matches {m.sum()} stations'
    i = stations.index[m][0]
    assert stations.at[i, 'phase'] == '', f'{r.station} has two rows'
    stations.at[i, 'phase'] = 'today' if r.phase == 'done' else PHASE_KEY[(r.phase, r.funded)]
    stations.at[i, 'year'] = int(r.year) if r.year else -1
assert (stations['phase'] != '').all(), stations.loc[stations['phase'] == '', 'name'].tolist()
stations['ada'] = stations['phase'].eq('today')
left = int((~stations['ada']).sum())
# the portal lists 144 stations; CTA counts 146 (State/Lake, closed for rebuilding, and the two Loop
# platforms it counts separately are not split here). The stations still to do must agree.
assert left == CTA_PUBLISHED['total'] - CTA_PUBLISHED['accessible'], f'{left} left, CTA says {CTA_PUBLISHED["total"] - CTA_PUBLISHED["accessible"]}'
log.info('Stations: %d, accessible today %d (portal flag %d + CTA releases), still to do %d', len(stations), stations['ada'].sum(), counts['accessible'], left)
stations.groupby('phase').size()

In [ ]:
# The four Red Line Extension stations: the alignment crossing each station's street
align = shapely.union_all(osm_rle.loc[osm_rle['railway'] == 'construction'].to_crs(walk.CRS).geometry)
rle_rows = []
for sid, short, street in RLE_STATIONS:
    s = shapely.union_all(osm_rle.loc[osm_rle['name'].fillna('').str.endswith(street) & osm_rle['highway'].notna()].to_crs(walk.CRS).geometry)
    hit = align.intersection(s)
    pts = [g for g in getattr(hit, 'geoms', [hit]) if g.geom_type == 'Point']
    assert pts, f'The alignment does not cross {street}'
    c = shapely.centroid(shapely.MultiPoint(pts))
    assert shapely.MultiPoint(pts).bounds[2] - shapely.MultiPoint(pts).bounds[0] < 120, f'{street}: crossings spread out'
    lon, lat = gpd.GeoSeries([c], crs=walk.CRS).to_crs(4326).get_coordinates().to_numpy()[0]
    rle_rows.append({'map_id': sid, 'name': f'{short} (Red Line Extension)', 'short': short, 'lines': ['Red'],
                     'lat': lat, 'lon': lon, 'flag': False, 'phase': 'RLE', 'year': 2030, 'ada': False})
stations = pd.concat([stations, pd.DataFrame(rle_rows)], ignore_index=True)
stations['x'], stations['y'] = to_address(stations['lat'], stations['lon'])
# the alignment, in address units, for the map (95th/Dan Ryan to 130th)
al_ll = gpd.GeoSeries([shapely.line_merge(align)], crs=walk.CRS).to_crs(4326).iloc[0]
al_addr = shapely.transform(al_ll, lambda c: np.column_stack(to_address(c[:, 1], c[:, 0])))
rle_path = [[[round(x), round(y)] for x, y in shapely.simplify(l, 8).coords] for l in getattr(al_addr, 'geoms', [al_addr])]
stations[stations['phase'] == 'RLE'][['short', 'lat', 'lon', 'x', 'y']]

In [ ]:
def utm(lat, lon) -> np.ndarray:
    pts = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs='EPSG:4326').to_crs(walk.CRS)
    return np.column_stack([pts.x, pts.y])


st_xy = utm(stations['lat'], stations['lon'])
bl_xy = utm(blocks['lat'], blocks['lon'])
stations['in_city'] = shapely.contains_xy(city, st_xy[:, 0], st_xy[:, 1])
ids = stations['map_id'].to_numpy()
P = int(blocks['pop'].sum())

area = gpd.GeoSeries([city.buffer(NETWORK_BUFFER_M)], crs=walk.CRS).to_crs('EPSG:4326').iloc[0]
xml = walk.extract(pbf, area, OSM_DIR / f'chicago-walk-{osm_date.isoformat()}.osm')
g_free = walk.main_component(walk.step_free(walk.load(xml)))
E_free = walk.Edges.of(g_free)
log.info('Step-free network: %d nodes, %d edges', len(E_free.nodes), len(E_free.length))

## 5. Analyze

Each step adds its stations to the accessible set and reruns the step-free walk (one multi-source
Dijkstra). A block gains when its walk drops to a half mile or less at that step; the gain goes to
the station it now walks to. The order matters only for how the gain is split between steps; the
total does not depend on it.

In [ ]:
def run(keep: np.ndarray, limit: float = WALK_M * 1.5):
    d, k, off = walk.reach(E_free, list(ids[keep]), st_xy[keep], limit=limit, max_snap=MAX_SNAP_M)
    bd, bk = walk.point_distance(E_free, d, k, bl_xy)
    return d, k, bd, bk, off


on = stations['ada'].to_numpy().copy()
d, k, bd, bk, _ = run(on, limit=WATER_LEVELS_MI[-1] * MILE_M + 800)  # far enough for the median walk
nodes = {'today': (d, k)}
blocks['d_today'], blocks['k_today'] = bd, bk
prev = bd
rows, gains = [], []
for key, label, year, funded in STEPS:
    on |= stations['phase'].eq(key).to_numpy()
    last = key == STEPS[-1][0]
    d, k, bd, bk, off = run(on, limit=WATER_LEVELS_MI[-1] * MILE_M + 800 if last else WALK_M * 1.5)
    new = (bd <= WALK_M) & (prev > WALK_M)
    blocks[f'd_{key}'], blocks[f'k_{key}'] = bd, bk
    g = blocks[new].groupby(pd.Series(bk[new], index=blocks.index[new]))[['pop', 'amb']].sum()
    gains.append(g.assign(step=key))
    rows.append({'key': key, 'label': label, 'year': year, 'funded': funded,
                 'stations': int(stations['phase'].eq(key).sum()),
                 'gain': int(blocks.loc[new, 'pop'].sum()), 'gainAmbulatory': float(blocks.loc[new, 'amb'].sum()),
                 'residents': int(blocks.loc[bd <= WALK_M, 'pop'].sum())})
    nodes[key] = (d, k)
    prev = bd
    if key == 'RLE':
        nodes['dated'] = (d, k)
steps = pd.DataFrame(rows)
d38, k38 = nodes['RPM']
blocks['d38'] = bd
assert np.isfinite(off[stations['phase'].eq('RLE').to_numpy()[on]]).all(), 'An extension station is off the network'
station_gain = pd.concat(gains).groupby(level=0)[['pop', 'amb']].sum()
assert station_gain['pop'].sum() == steps['gain'].sum()
steps

In [ ]:
today = int(blocks.loc[blocks['d_today'] <= WALK_M, 'pop'].sum())
dated = int(steps.loc[steps['key'] == 'RLE', 'residents'].iloc[0])
full = int(steps['residents'].iloc[-1])
gain = full - today
nodate = full - dated
assert steps['gain'].sum() == gain


def wmedian(d: pd.Series) -> float:
    o = d.sort_values()
    c = blocks.loc[o.index, 'pop'].cumsum() / P
    return float(np.interp(0.5, c, o)) / MILE_M


summary = {
    'residents': P,
    'stationsToday': int(stations['ada'].sum()),
    'stationsLeft': left,
    'stationsUndated': int(stations['phase'].isin(['P2u', 'P3', 'P4', 'RPM']).sum()),
    'today': today, 'todayShare': today / P,
    'dated': dated, 'datedShare': dated / P,
    'full': full, 'fullShare': full / P,
    'gain': gain, 'datedGain': dated - today, 'noDateGain': nodate, 'noDateShare': nodate / gain,
    'beyond': P - full, 'beyondShare': (P - full) / P,
    'withinMileShare': float(blocks.loc[blocks['d38'] <= MILE_M, 'pop'].sum() / P),
    'withinTwoMilesShare': float(blocks.loc[blocks['d38'] <= 2 * MILE_M, 'pop'].sum() / P),
    'medianWalkTodayMi': wmedian(blocks['d_today']), 'medianWalk2038Mi': wmedian(blocks['d38']),
    'ambulatory': float(blocks['amb'].sum()),
    'ambulatoryToday': float(blocks.loc[blocks['d_today'] <= WALK_M, 'amb'].sum()),
    'ambulatoryFull': float(blocks.loc[blocks['d38'] <= WALK_M, 'amb'].sum()),
    'rleGain': int(steps.loc[steps['key'] == 'RLE', 'gain'].iloc[0]),
}
# where the no-date gain lives: community areas
undated_new = (blocks['d38'] <= WALK_M) & (blocks['d_RLE'] > WALK_M)
by_ca = blocks[undated_new].groupby('commarea')['pop'].sum().sort_values(ascending=False)
log.info('Summary: %s', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in summary.items()})
by_ca.head(8)

In [ ]:
# The finding printed on the plate, generated from the data. Rounded once: thousands.
th = lambda n: f'{round(n, -3):,.0f}'
pc = lambda v: f'{round(100 * v):.0f}%'
FINDING = (
    f'If every ‘L’ station were accessible, {pc(summary["fullShare"])} of Chicagoans would live within a step-free '
    f'half-mile walk of one, up from {pc(summary["todayShare"])} today. '
    f'But {pc(summary["noDateShare"])} of that gain waits on stations with no date or no money. '
    f'Even then, {pc(summary["beyondShare"])} of the city lives farther away.'
)
print(FINDING)

## 6. Polygons

**Strata.** Three nested walksheds: today, after the dated steps (through the Red Line Extension), and
the whole promise. The plate draws the whole promise hatched, the dated one stippled over it and
today solid on top. **Waterlines.** The step-free distance to the nearest station in 2038, on a
40 m raster (each cell takes the nearest of six walk-network nodes, plus the straight walk to it),
lightly smoothed and contoured every eighth of a mile from 5/8 mile to 4 miles, clipped to the city.

In [ ]:
def shed_of(key):
    d, k = nodes[key]
    return walk.shed(walk.reached(E_free, d, k, WALK_M).geometry, FRONTAGE_M, MIN_HOLE_M2).intersection(city)


sheds = {'today': shed_of('today'), 'dated': shed_of('dated'), 'full': shed_of('RPM')}
assert sheds['today'].difference(sheds['dated']).area < 1e3 and sheds['dated'].difference(sheds['full']).area < 1e3, 'Strata are not nested'
SQ_MI = 2_589_988.11
areas = {f'{k}SqMi': v.area / SQ_MI for k, v in sheds.items()} | {'citySqMi': city.area / SQ_MI}
rings = {k: walk.to_rings(v) for k, v in sheds.items()}
log.info('Areas (sq mi): %s', {k: round(v, 1) for k, v in areas.items()})

In [ ]:
# waterlines
nxy = np.array([(g_free.nodes[n]['x'], g_free.nodes[n]['y']) for n in E_free.nodes])
ok = np.isfinite(d38)
tree = cKDTree(nxy[ok])
x0, y0, x1, y1 = city.bounds
gx = np.arange(x0 - 400, x1 + 400, WATER_CELL_M)
gy = np.arange(y0 - 400, y1 + 400, WATER_CELL_M)
GX, GY = np.meshgrid(gx, gy)
dd, ii = tree.query(np.c_[GX.ravel(), GY.ravel()], k=6)
field = (np.min(d38[ok][ii] + dd, axis=1).reshape(GX.shape)) / MILE_M
field = gaussian_filter(field, 1.5)
water = {}
for lv in WATER_LEVELS_MI:
    lines = [shapely.LineString(np.c_[gx[0] + c[:, 1] * WATER_CELL_M, gy[0] + c[:, 0] * WATER_CELL_M])
             for c in measure.find_contours(field, lv) if len(c) > 6]
    if not lines:
        continue
    clipped = shapely.intersection(shapely.MultiLineString(lines), city)
    ll = gpd.GeoSeries([clipped], crs=walk.CRS).to_crs(4326).iloc[0]
    addr = shapely.transform(ll, lambda c: np.column_stack(to_address(c[:, 1], c[:, 0])))
    addr = shapely.simplify(addr, 6)
    parts = [l for l in getattr(addr, 'geoms', [addr]) if l.geom_type == 'LineString' and l.length > 60]
    water[f'{lv:.3f}'] = [[[round(x), round(y)] for x, y in l.coords] for l in parts]
log.info('Waterlines: %d levels, %d vertices', len(water), sum(len(l) for v in water.values() for l in v))
# check: a block's own distance and the field agree (median absolute difference, miles)
fb = field[np.clip(((bl_xy[:, 1] - gy[0]) / WATER_CELL_M).round().astype(int), 0, len(gy) - 1),
           np.clip(((bl_xy[:, 0] - gx[0]) / WATER_CELL_M).round().astype(int), 0, len(gx) - 1)]
field_err = float(np.median(np.abs(fb - blocks['d38'].to_numpy() / MILE_M)))
log.info('Field vs block distance: median |diff| %.3f mi', field_err)
assert field_err < 0.05

In [ ]:
# The inset: the largest piece of the no-date stratum, by the residents who gain there
undated = sheds['full'].difference(sheds['dated'])
parts = gpd.GeoDataFrame(geometry=list(getattr(undated, 'geoms', [undated])), crs=walk.CRS)
pts = gpd.GeoDataFrame(blocks.loc[undated_new, ['pop']], geometry=gpd.points_from_xy(*bl_xy[undated_new.to_numpy()].T), crs=walk.CRS)
hits = gpd.sjoin(pts, parts, predicate='within', how='left')
part_pop = hits.groupby('index_right')['pop'].sum().sort_values(ascending=False)
largest = parts.geometry[part_pop.index[0]]
lb = gpd.GeoSeries([largest], crs=walk.CRS).to_crs('EPSG:4326').total_bounds
ix, iy = to_address([lb[1], lb[3]], [lb[0], lb[2]])
inset = {'box': {'west': float(ix[0]), 'east': float(ix[1]), 'south': float(iy[0]), 'north': float(iy[1])},
         'residents': int(part_pop.iloc[0]),
         'stations': stations.loc[stations['map_id'].isin(blocks.loc[undated_new].loc[hits.index[hits['index_right'] == part_pop.index[0]], 'k_RPM']), 'short'].tolist()}
log.info('Inset: %s', inset)

## 7. QA view

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
gpd.GeoSeries([city], crs=walk.CRS).plot(ax=ax, color='0.94')
gpd.GeoSeries([sheds['full']], crs=walk.CRS).plot(ax=ax, facecolor='none', hatch='////', edgecolor='0.2', linewidth=0)
gpd.GeoSeries([sheds['dated']], crs=walk.CRS).plot(ax=ax, color='0.8')
gpd.GeoSeries([sheds['today']], crs=walk.CRS).plot(ax=ax, color='0.6')
cs = ax.contour(GX, GY, field, levels=[0.75, 1, 1.5, 2, 3], colors='0.4', linewidths=0.4)
for ph, mk in (('today', 'o'), ('RLE', 's')):
    m = stations['phase'].eq(ph).to_numpy()
    ax.scatter(*st_xy[m].T, s=6 if ph == 'today' else 20, color='0.1', marker=mk, zorder=3)
m = ~stations['ada'].to_numpy() & ~stations['phase'].eq('RLE').to_numpy()
ax.scatter(*st_xy[m].T, s=14, facecolor='white', edgecolor='0.1', zorder=3)
gpd.GeoSeries([largest], crs=walk.CRS).boundary.plot(ax=ax, color='red', linewidth=0.8)
minx, miny, maxx, maxy = city.bounds
ax.set(xlim=(minx, maxx), ylim=(miny, maxy), title='Today (dark), dated (light), no date (hatched)')
ax.set_axis_off()
plt.show()
stations.assign(gain=stations['map_id'].map(station_gain['pop']).fillna(0).astype(int)) \
    .query('phase != "today"').sort_values('gain', ascending=False)[['short', 'lines', 'phase', 'year', 'gain']].head(15)

## 8. Export

`plate.json`: ledger, finding, summary, steps, stations (with phase, year and gain), the three strata,
waterlines, the Red Line Extension path, the 'L' track and the inset. Keys are camelCase.

In [ ]:
ledger = Ledger(
    source=STOPS_NAME,
    dataset_id=STOPS_ID,
    url=STOPS_URL,
    pulled=stops_pulled,
    units=f'{P:,} residents (2020 Census) in {len(blocks):,} blocks; {counts["total"]} stations + 4 under construction',
    method=[
        'Phases from CTA’s ASAP Strategic Plan (2024), Table 5; Racine (Blue) and Austin (Green) accessible per CTA releases',
        'Red Line Extension stations where OpenStreetMap’s construction alignment crosses CTA’s cross streets',
        f'Half-mile step-free walk on the OpenStreetMap network (extract of {osm_date}), as Plate III, rerun after each step',
        'Waterlines: step-free walk to the nearest station in 2038, every eighth of a mile',
    ],
    caveats=[
        '2020 population for a 2038 city; today’s sidewalks and curb ramps',
        'Phases Three and Four and the Red–Purple stations have no date or funding; the order among them is CTA’s table order',
        'Extension station points are approximate (main entrance street, not the entrance)',
    ],
)
r1 = lambda v: round(float(v), 1)
phase_label = {k: lab for k, lab, _, _ in STEPS} | {'today': 'Accessible today'}
payload = {
    'source': {'name': STOPS_NAME, 'id': STOPS_ID, 'url': STOPS_URL, 'pulled': stops_pulled.isoformat()},
    'asap': {'plan': ASAP_PLAN, 'page': ASAP_PAGE, 'rleFaq': RLE_FAQ},
    'osm': {'extract': osm_date.isoformat(), 'source': 'Geofabrik, Illinois'},
    'gtfs': {'feedDate': gtfs_date.isoformat(), 'url': gtfs.FEED_URL},
    'census': {'blocks': len(blocks), 'boundaryPulled': boundary_pulled},
    'walkM': round(WALK_M, 1),
    'summary': {**summary, **areas, 'fieldErrorMi': field_err},
    'steps': rows,  # the records, so a missing year stays None (null), not NaN
    'stations': [
        {'id': r.map_id, 'short': r.short, 'name': r.name, 'lines': r.lines, 'ada': bool(r.ada),
         'phase': r.phase, 'phaseLabel': phase_label[r.phase], 'year': int(r.year) if r.year > 0 else None,
         'x': r1(r.x), 'y': r1(r.y), 'inCity': bool(r.in_city),
         'gain': int(station_gain['pop'].get(r.map_id, 0)),
         'residents': int(blocks.loc[(blocks['d38'] <= WALK_M) & (blocks['k_RPM'] == r.map_id), 'pop'].sum())}
        for r in stations.itertuples()
    ],
    'rail': track,
    'rle': rle_path,
    'polygons': rings,
    'water': water,
    'inset': inset,
}
path = write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)
assert 'NaN' not in path.read_text(), 'NaN is not valid JSON for the site'
log.info('Wrote %s (%.0f KB)', path.relative_to(DAY_DIR), path.stat().st_size / 1024)